## 대화형 챗봇 (메모리)

### 🔹 Conversation Memory란?

> **Conversation Memory(대화 메모리)** 는 LLM이 **이전 대화 내용을 기억하고 이어서 대답**할 수 있도록 하는 기능입니다.

일반적인 LLM은 한 번의 요청(prompt)만 처리하고 끝나기 때문에,  
사용자가 “아까 말한 여행지 일정 다시 알려줘.”처럼 과거 대화를 참조하면 맥락을 잃어버립니다.  
→ 이때 필요한 것이 바로 **Memory(기억 기능)** 입니다.


### 🔹 왜 Memory가 필요할까?

| 상황                                               | Memory 없을 때                                                          | Memory 있을 때                                    |
| -------------------------------------------------- | ----------------------------------------------------------------------- | ------------------------------------------------- |
| 사용자: “부산 여행지 추천해줘.”<br>→ 모델이 답변함 | 다음 질문: “그럼 거기 근처 맛집은?”<br>→ 모델이 “어디 여행 말씀인가요?” | “부산 근처엔 회센터가 많고… 홍합탕집이 유명해요!” |
| 사용자: “어제 추천해준 책 제목 다시 말해줘.”       | “어떤 책을 말씀하시는 건가요?”                                          | “어제 말씀드린 건 『데미안』이에요.”              |

👉 메모리를 활용하면,  
모델이 “과거 대화의 맥락(Context)”을 유지한 채로  
**자연스러운 멀티턴(Multi-turn) 대화형 챗봇**을 만들 수 있습니다.


### 🔹 LangChain에서의 Memory 개념

LangChain은 다양한 형태의 “기억 클래스(memory classes)”를 제공합니다.  
대표적으로 아래 세 가지를 이해하면 충분합니다 👇

| Memory 클래스                        | 설명                            | 특징                                             |
| ------------------------------------ | ------------------------------- | ------------------------------------------------ |
| **`ConversationBufferMemory`**       | 단순히 대화 전체를 계속 저장    | 가장 기본적이고 직관적                           |
| **`ConversationBufferWindowMemory`** | 최근 N개의 대화만 저장          | “단기 기억” 형태로, 긴 대화를 효율적으로 유지    |
| **`ConversationSummaryMemory`**      | 이전 대화를 LLM이 요약해서 저장 | 긴 대화를 핵심 요약본으로 관리, 맥락 유지에 적합 |


### 🔹 프롬프트에 직접 넣는 방식 vs Memory 클래스 사용 비교

| 비교 항목   | 프롬프트에 직접 삽입                                                        | Memory 클래스 사용                     |
| ----------- | --------------------------------------------------------------------------- | -------------------------------------- |
| 코드 복잡도 | ❌ 대화마다 프롬프트를 새로 생성해야 함                                     | ✅ LangChain이 자동으로 이전 대화 삽입 |
| 확장성      | ❌ 대화가 길어지면 토큰 초과 문제 발생                                      | ✅ 오래된 대화는 요약/제한 가능        |
| 유지보수    | ⚠️ 과거 대화 삽입 위치만 교체하면 되지만, 요약·토큰 제어를 직접 구현해야 함 | ✅ 메모리 객체만 관리하면 됨           |
| 현실성      | ❌ “일회용 챗봇”에만 적합                                                   | ✅ “지속형 대화형 챗봇” 구현 가능      |

즉, **Memory는 LLM이 “대화 히스토리를 스스로 관리”하도록 하는 스마트한 도우미 클래스**입니다.


### 🔹 Memory 작동 원리

1. 사용자가 LLM에 **질문(입력)** 을 보냅니다.

2. LLM이 **답변**을 생성하면, 이 대화 내용이 자동으로 **Memory에 저장**됩니다.
3. 사용자가 다음 질문을 입력하면,  
   LangChain이 **이전 대화 기록을 자동으로 프롬프트에 포함시켜** LLM에 전달합니다.
4. LLM은 이 정보를 바탕으로 **맥락을 이해하고 연속적인 대화**를 생성합니다.

→ 즉, 사용자는 별도로 과거 대화를 넣을 필요 없이, LangChain이 **“대화 기록 → 프롬프트 삽입 → LLM 호출”**  
과정을 자동으로 처리합니다.


In [1]:
#!uv add -U langchain langchain-openai  python-dotenv tiktoken

### 코드 예시 1. 기본 Buffer Memory (대화 전체 저장)


In [3]:
### 코드 예시 ① 기본 Buffer Memory
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 모델 선언
llm = ChatOpenAI(model="gpt-5-nano")

# 메모리 객체 생성 (대화 전체를 버퍼 형태로 저장)
memory = ConversationBufferMemory(return_messages=True)

# 프롬프트 템플릿
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 여행 전문가야. 사용자의 질문에 친절하게 답해줘."),
        MessagesPlaceholder(variable_name="history"),  # 이전 대화 삽입 위치
        ("human", "{input}"),  # 사용자 질문
    ]
)

# LCEL 체인 구성
chain = prompt | llm

# 연속 대화 시뮬레이션
inputs = ["부산 여행지 추천해줘.", "그럼 그 근처 맛집은 어디야?"]

for user_input in inputs:
    # 메모리 불러오기
    history = memory.load_memory_variables({})["history"]
    # 실행
    result = chain.invoke({"history": history, "input": user_input})
    # 결과 출력
    print(f"\n사용자: {user_input}\n 응답: {result.content}")
    # 메모리에 저장
    memory.save_context({"input": user_input}, {"output": result.content})

/Users/parkchanryong/Desktop/SK네트웍스/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/cn/n1wp5rkx27j7415_2rb44sk00000gn/T/ipykernel_14353/641154472.py:10: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(return_messages=True)



사용자: 부산 여행지 추천해줘.
 응답: 좋아요! 부산은 바다와 산, 맛집이 한꺼번에 있는 매력적인 도시예요. 관심사에 따라 맞춤 코스도 가능하지만, 먼저 꼭 가보면 좋은 대표 명소들을 정리해볼게요.

핵심 명소 (카테고리별)
- 해운대 해수욕장 + 동백섬: 부산을 대표하는 해변 말 그대로의 멋짐. 산책로와 전망 좋은 카페 많고, 여름철은 해수욕 파티 분위기!
- 광안리 해변 + 광안대교 야경: 해가 지고 나면 바다가 은은하게 빛나고 다리도 멋진 포토 포인트.
- BIFF 광장 + 남포동 국제시장: 쇼핑 + 맛집 골목. 영화를 좋아한다면 BIFF 거리의 분위기를 만끽하기 좋습니다.
- 자갈치 시장: 싱싱한 해산물 체험과 현지 식문화 맛보기. 회, 조개구이, 생선구이 등 다양해요.
- 감천문화마을: 알록달록 색감의 언덕 마을, 전망대에서 사진 잘 나오는 포인트 다수.
- 태종대: 해안 절벽 산책로와 등대. 파도 소리와 바다 풍경이 멋져요.
- 해동용궁사: 바다를 바라보는 옛스러운 사찰 풍경이 독특합니다.
- 금정산/범어사: 산책로와 고즈넉한 사찰 체험. 도심에서 벗어나 한숨 돌리기 좋습니다.
- 부산타워(용두산공원): 도심에서 시내 전경을 한눈에 볼 수 있는 전망 포인트.
- 센텀시티: 세계 최대 규모의 백화점 단지. 쇼핑과 식사, 문화 공간까지 한곳에서 해결 가능.
- 해양생태 체험: SEA LIFE 부산아쿠아리움 등 해양 테마 체험도 가족 단위에 좋습니다.
- 근교 해변들: 기장 일광/정관 해변 등도 조용하게 바다를 즐길 수 있어요.

2-3일 코스 예시
- 2박3일 코스 (전형적인 풍으로 구성)
  - Day 1: 남포동/BIFF 광장 → 국제시장 → 자갈치 시장 → 감천문화마을
  - Day 2: 해운대 해수욕장 → 동백섬 산책로 → 달맞이길 카페 거리 → 광안리 야경
  - Day 3: 태종대 혹은 해동용궁사 방문 → 센텀시티에서 점심/쇼핑으로 마무리
- 1박2일 코스 (도심 집중)
  - Day 1: 남포동/BIFF 광장 → 자갈치 시장 → 감천문화

모델은 앞선 “부산 여행” 대화를 기억하고 자연스럽게 이어서 대답합니다.


In [4]:
memory.load_memory_variables({})["history"]

[HumanMessage(content='부산 여행지 추천해줘.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='좋아요! 부산은 바다와 산, 맛집이 한꺼번에 있는 매력적인 도시예요. 관심사에 따라 맞춤 코스도 가능하지만, 먼저 꼭 가보면 좋은 대표 명소들을 정리해볼게요.\n\n핵심 명소 (카테고리별)\n- 해운대 해수욕장 + 동백섬: 부산을 대표하는 해변 말 그대로의 멋짐. 산책로와 전망 좋은 카페 많고, 여름철은 해수욕 파티 분위기!\n- 광안리 해변 + 광안대교 야경: 해가 지고 나면 바다가 은은하게 빛나고 다리도 멋진 포토 포인트.\n- BIFF 광장 + 남포동 국제시장: 쇼핑 + 맛집 골목. 영화를 좋아한다면 BIFF 거리의 분위기를 만끽하기 좋습니다.\n- 자갈치 시장: 싱싱한 해산물 체험과 현지 식문화 맛보기. 회, 조개구이, 생선구이 등 다양해요.\n- 감천문화마을: 알록달록 색감의 언덕 마을, 전망대에서 사진 잘 나오는 포인트 다수.\n- 태종대: 해안 절벽 산책로와 등대. 파도 소리와 바다 풍경이 멋져요.\n- 해동용궁사: 바다를 바라보는 옛스러운 사찰 풍경이 독특합니다.\n- 금정산/범어사: 산책로와 고즈넉한 사찰 체험. 도심에서 벗어나 한숨 돌리기 좋습니다.\n- 부산타워(용두산공원): 도심에서 시내 전경을 한눈에 볼 수 있는 전망 포인트.\n- 센텀시티: 세계 최대 규모의 백화점 단지. 쇼핑과 식사, 문화 공간까지 한곳에서 해결 가능.\n- 해양생태 체험: SEA LIFE 부산아쿠아리움 등 해양 테마 체험도 가족 단위에 좋습니다.\n- 근교 해변들: 기장 일광/정관 해변 등도 조용하게 바다를 즐길 수 있어요.\n\n2-3일 코스 예시\n- 2박3일 코스 (전형적인 풍으로 구성)\n  - Day 1: 남포동/BIFF 광장 → 국제시장 → 자갈치 시장 → 감천문화마을\n  - Day 2: 해운대 해수욕장 → 동백섬 산책로 → 달맞이길 카페 거리 → 광안리 야경\n  - D

### 코드 예시 2. 최근 대화만 유지 (Window Memory)


In [5]:
### 코드 예시 ① 기본 Buffer Memory
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 모델 선언
llm = ChatOpenAI(model="gpt-5-nano")

# 메모리 객체 생성 (대화 전체를 버퍼 형태로 저장)
memory = ConversationBufferWindowMemory(k=2, return_messages=True)

# 프롬프트 템플릿
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 여행 전문가야. 사용자의 질문에 친절하게 답해줘."),
        MessagesPlaceholder(variable_name="history"),  # 이전 대화 삽입 위치
        ("human", "{input}"),  # 사용자 질문
    ]
)

# LCEL 체인 구성
chain = prompt | llm

# 5️⃣ 연속 대화 시뮬레이션
inputs = [
    "부산 여행지 추천해줘.",
    "대한 민국의 수도는 어디야?",
    "서울의 인구는 몇명이야?",
    "내가 아까 추천해달라고 한 어행지는 어디야?",
]

for user_input in inputs:
    # 메모리 불러오기
    history = memory.load_memory_variables({})["history"]
    # 실행
    result = chain.invoke({"history": history, "input": user_input})
    # 결과 출력
    print(f"\n사용자: {user_input}\n 응답: {result.content}")
    # 메모리에 저장
    memory.save_context({"input": user_input}, {"output": result.content})


/var/folders/cn/n1wp5rkx27j7415_2rb44sk00000gn/T/ipykernel_14353/3428620564.py:10: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferWindowMemory(k=2, return_messages=True)



사용자: 부산 여행지 추천해줘.
 응답: 좋아요! 부산은 해변, 시장, 문화마을이 한꺼번에 즐길 수 있어서 매력이 크죠. 관심사나 일정이 있다면 알려주면 맞춤으로 더 구체화해드릴게요. 우선 대표적인 명소와 간단한 일정 제안을 드릴게요.

추천 장소 모음
- 해운대 해수욕장 + 동백섬: 부산의 대표 해변, 산책로와 작은 공원이 많아 낮에 걷기 좋아요. 여름에는 해수욕도 가능.
- 달맞이길: 해안선 따라 드라이브나 산책하기 좋은 코스. 카페와 맛집도 많아요.
- 광안리 해수욕장 & 광안대교 야경: 야경이 especially 멋져요. 더베이101 같은 루프탑 카페도 많습니다.
- 자갈치시장 + 국제시장: 신선한 해산물 구경하고 해산물 요리 맛보기 좋고, 시장 골목의 분위기를 제대로 느껴볼 수 있어요.
- BIFF 광장: 부산국제영화제의 중심지 느낌. 영화관 분위기와 카페가 많아요.
- 감천문화마을: 알록달록한 계단길과 벽화가 인상적인 문화마을. 사진 찍기 좋아요.
- 태종대 / 등대 전망길: 바다 절벽을 따라 걷기 좋은 코스. 전망이 매우 좋습니다.
- 해동용궁사: 바다를 바라보며 절을 감상하는 절경 명소.
- 송도해수욕장 + 누리마루 APec 하우스: 멋진 바다 풍경과 함께 산책하기 좋아요. 인스타 핫스팟이 많습니다.
- 오륙도 스카이워크: 바다 아래를 유리 바닥으로 걷는 독특한 체험.
- 범어사 / 금정산: 도심에서 가볍게 하는 산책이나 하이킹으로 피로를 풀 수 있어요.
- 센텀시티 & 신세계 센텀시티 스파랜드: 쇼핑과 피로 해소를 한꺼번에 원하면 강추. 대형 쇼핑몰과 스파.

초보자용 2박 3일 코스 제안
- Day 1: 남포동/자갈치시장 → BIFF광장 산책 → 감천문화마을 → 광안리 야경
  - 오전: 자갈치시장 방문해 신선한 해산물 간단 간식
  - 점심: 국제시장 근처 맛집에서 지역 음식 체험
  - 오후: 감천문화마을 산책, 인생샷 스팟 탐방
  - 저녁: 광안리에서 해변 산책 후 광안대교 야경 감상
- Day 2: 해운대 지역 집중
  - 오전: 

이 방식은 단기 기억처럼 작동합니다.  
오래된 대화는 자동으로 지워져 토큰 낭비를 줄이고 효율성을 높입니다.


### 코드 예시 3. 이전 대화를 요약해서 저장 (ConversationSummaryMemory)


In [6]:
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationSummaryMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 모델 선언
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7)

# 요약형 메모리 생성 — LLM이 과거 대화를 요약해 맥락을 유지
memory = ConversationSummaryMemory(llm=llm, return_messages=True)

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "너는 여행 플래너야. 사용자의 요구에 따라 가족 여행 일정을 제안해줘.",
        ),
        MessagesPlaceholder(variable_name="history"),  # 요약된 과거 대화 자동 삽입
        ("human", "{input}"),  # 현재 질문
    ]
)

# LCEL 체인 구성
chain = prompt | llm

# 연속 대화 시뮬레이션
inputs = [
    "이번 주말에 가족 여행지 추천해줘.",
    "지난번에 추천한 곳 중에 아이들이 놀기 좋은 곳은 어디였지?",
    "그럼 거기 일정표를 하루만 짜줘.",
]

for user_input in inputs:
    history = memory.load_memory_variables({})["history"]
    result = chain.invoke({"history": history, "input": user_input})
    print(f"\n👤 사용자: {user_input}\n🤖 응답: {result.content}")
    memory.save_context({"input": user_input}, {"output": result.content})

/var/folders/cn/n1wp5rkx27j7415_2rb44sk00000gn/T/ipykernel_14353/428031871.py:9: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationSummaryMemory(llm=llm, return_messages=True)



👤 사용자: 이번 주말에 가족 여행지 추천해줘.
🤖 응답: 좋아요! 이번 주말 가족 여행을 쉽게 시작할 수 있도록 수도권에서 가볍게 다녀올 만한 가족 친화적인 코스 네 가지를 뽑아봤어요. 원하시면 출발지(서울/수원 등), 인원 수, 예산, 아이들 연령대에 맞춰 더 구체적인 일정으로 맞춰드릴게요.

1) 가평/청평 계곡&수상레저 (2일)
- 왜 좋나요: 도시 근교에서 자연을 느끼고, 아이들도 좋아하는 물놀이와 숲놀이가 모두 가능해요. 차로 1–2시간대 거리라 부담이 적어요.
- 대표 포인트: 청평호반 산책, 계곡 물놀이, 가족 펜션이나 리조트 숙박, 수상레저 체험 가능.
- 간단한 일정 예시
  - 토: 서울 출발 → 가평 도착(오전) -> 청평호수 산책/자연체험 -> 점심 -> 수상레저 체험(보트/수상스키) -> 펜션 체크인
  - 일: 아침 산책 -> 모험형 숲 체험 코스 -> 점심 -> 서울로 복귀
- 비 오는 날 대체: 계곡 인근 카페/전시관 방문, 실내 놀이시설, 펜션 내 보드게임.

2) 강릉 1박 2일 (바다+카페거리)
- 왜 좋나요: 바다 냄새 맡으며 걷고, 아이와 커피거리도 즐길 수 있고 날씨가 맑으면 해변에서 신나게 놀기 좋아요.
- 대표 포인트: 경포해변 산책, 안목 커피거리, 주문진 해산물시장, 근처에 있는 짧은 산책로나 공원
- 간단한 일정 예시
  - 토: 서울/수도권 출발 → 강릉 도착 -> 경포 해변 산책/모래놀이 -> 점심 -> 안목 커피거리 카페 투어 -> 숙소 체크인
  - 일: 아침 해변 산책 -> 정동진/선유도 트레일 짧은 코스(선택) -> 점심 -> 서울로 복귀
- 비 오는 날 대체: 커피거리 맛집 탐방, 실내 수족관/전시관 방문.

3) 속초/설악산 1박 2일 (산+바다)
- 왜 좋나요: 아이들이 자연 탐험을 좋아하면 설악산과 바다가 한꺼번에 즐길 수 있어요. 케이블카 타고 가볍게 산책도 가능.
- 대표 포인트: 설악산 권역의 케이블카, 속초 중앙시장, 속초 아바이마을의 해물요리, 속초 해수욕장
- 간단한 일정 예시
 

In [7]:
# 현재까지의 요약본 확인
print(memory.buffer)

새로운 요약: 사용자가 하루 일정표를 요청하자 AI는 앞서 제시된 네 가지 아이들 친화 일정 중 하나를 골라 하루치 일정으로 간단히 구성해 드리겠다고 제안했고, 각 옵션에 대해 시간표 형태의 샘플(출발 시각 포함), 아이의 추천 연령대, 비 오는 날 대안, 간단 팁을 제시했다. 옵션 1은 Gapyeong/Cheongpyeong 물놀이 테마, 2는 Gangneung 해변+카페거리, 3은 Sokcho/Seoraksan 근처 산+바다 맛보기, 4는 Paju DMZ+Provence Village로 구성되었으며, 모두 08:00~17:00 같은 하루 일정의 예시가 포함되었다. 또한 출발지, 인원, 예산에 맞춰 더 구체적인 1일 일정표(시간표, 동선, 식당 추천, 교통 수단)를 바로 맞춤 제작해 드릴 수 있고, 한 가지 옵션에 집중해 드릴 수도 있다는 점도 안내했다.


이 방식은 LLM이 이전 대화를 스스로 요약하여 핵심 맥락을 유지합니다.  
수백 문장의 대화도 핵심 내용만 남기기 때문에 효율적입니다.


**핵심 정리**
| Memory 타입 | 특징 | 사용 목적 |
|--------------|--------|-------------|
| **BufferMemory** | 모든 대화를 그대로 저장 | 짧은 대화, 디버깅용 |
| **WindowMemory** | 최근 N개의 대화만 유지 | 실시간 채팅, 효율성 |
| **SummaryMemory** | 이전 대화를 요약해서 유지 | 긴 대화 맥락 유지형 챗봇 |


🔹 실습 : 아래 조건을 만족하는 “나만의 대화형 여행 상담 챗봇”을 만들어보세요.

당신은 AI 여행 비서 트래블GPT 입니다.  
고객이 여러 도시를 순서대로 여행하면서 맞춤 일정·숙소·음식을 요청할 때,  
이전 대화 내용을 기억하며 연결된 제안을 해주는 지능형 여행 어시스턴트를 구현하세요.

1. ChatOpenAI(model="gpt-5-nano") 사용
2. ConversationSummaryMemory로 요약 기반 대화 기억 구현
3. 답변에 반드시 "이전 여행 내용을 바탕으로 추천드리면..." 이라는 문구 포함

- 예시 : 이전 여행 내용을 바탕으로 추천드리면, 여수에서는 해상케이블카와 낭만포차거리를 꼭 가보세요.

4. 대화 시나리오:

- 사용자가 “부산 → 여수 → 강릉” 순으로 도시를 이동
- 챗봇은 이전 도시에서 한 활동을 기억하고 “연결된 여행 루트”나 “테마별 추천(가족/커플/힐링)”을 제안할 것

예시 시나리오

```bash
사용자: 이번 주말엔 부산 갈 건데 가족 여행지 좀 추천해줘.
AI: 부산의 해운대, 아쿠아리움이 가족 단위로 인기예요!

사용자: 이번엔 여수로 가볼까?
AI: 이전 여행 내용을 바탕으로 추천드리면, 부산의 해변 감성에 이어 여수에서는 바다 전망 케이블카와 낭만포차를 즐기세요.

사용자: 그럼 마지막은 강릉이 좋을까?
AI: 이전 여행 내용을 바탕으로 추천드리면, 강릉에서는 여수보다 조용한 힐링 카페 거리와 바다 일출 코스를 권합니다.
```


In [3]:
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationSummaryMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 1. 모델 선언
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7)

# 2. 요약형 메모리 생성
memory = ConversationSummaryMemory(llm=llm, return_messages=True)

# 3. 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 AI 여행 비서 트래블GPT 입니다. 고객이 여러 도시를 순서대로 여행하면서 맞춤 일정·숙소·음식을 요청할 때, 이전 대화 내용을 기억하며 연결된 제안을 해주는 지능형 여행 어시스턴트입니다. 답변에 반드시 이전 여행 내용을 바탕으로 추천드리면...이라는 문구 포함",
        ),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)

# 4. LCEL 체인 구성
chain = prompt | llm

# 5. 대화 시나리오
inputs = [
    "이번 주말엔 부산 갈 건데 가족 여행지 좀 추천해줘.",
    "이번엔 여수로 가볼까?",
    "그럼 마지막은 강릉이 좋을까?",
]

# 6. 연속 대화 실행
for user_input in inputs:
    history = memory.load_memory_variables({})["history"]
    result = chain.invoke({"history": history, "input": user_input})
    print(f"\n사용자: {user_input}\n트래블GPT: {result.content}")
    memory.save_context({"input": user_input}, {"output": result.content})

# 7. 선택: 요약된 메모리 확인
print(memory.buffer)


사용자: 이번 주말엔 부산 갈 건데 가족 여행지 좀 추천해줘.
트래블GPT: 답변에 반드시 이전 여행 내용을 바탕으로 추천드리면... 현재 대화에 이전 여행 정보가 없으니, 부산에서 이번 주말 가족 여행에 적합한 일반적인 추천을 드립니다. 아이들 동선 고려해 손쉬운 루트를 중심으로 정리할게요.

1) 해운대 집중 코스 – 해운대 해수욕장 + 동백섬 산책 + 부산 아쿠아리움 + 광안리 야경
- 추천 이유: 해운대는 가족 친구성 높은 해변과 아쿠아리움이 있어 아이들이 좋아하고 이동 동선이 비교적 편합니다.
- 하루 일정 예시
  - 오전: 해운대 해수욕장 산책 및 모래 놀이(아이들) → 해운대 근처 카페 또는 맛집에서 점심
  - 오후: 부산 아쿠아리움 방문(Sea Life Busan) → 동백섬 산책로 따라 산책 및 누리마루 APEC 하우스 바다 전망 구경
  - 저녁: 광안리로 이동해 광안대교 야경 감상 및 광안리 해변 산책
- 팁: 어린 아이가 많다면 해운대역/해운대 해수욕장 인근 주차와 접근이 편한 곳을 선택하세요. 날씨가 좋지 않으면 아쿠아움이 훌륭한 대안이 됩니다.

2) 자연·바다 풍경 코스 – 태종대 공원과 송도 해상 케이블카
- 추천 이유: 짧은 산책로와 바다 풍경으로 아이들이 신나게 뛰놀 수 있고, 가족 단위로도 비교적 무리가 적습니다.
- 하루 일정 예시
  - 오전: 태종대 공원 도착 후 등대 전망대까지 짧은 트레일 산책(사진 찍기 좋음)
  - 점심: 영도 방향으로 간단히 식사 또는 해산물
  - 오후: 송도 해상 케이블카 타고 바다 위에서 광활한 풍경 감상 → 광안리나 해운대 쪽으로 이동해 가족 저녁
- 팁: 태종대는 코스에 따라 오르내림이 있어 유모차 사용 여부를 미리 체크하세요. 도보가 불편한 분은 케이블카와 전망대 위주로 루트를 조정하면 좋습니다.

3) 문화·시장 코스 – 감천문화마을 + 자갈치시장/국제시장 + 용두산 공원
- 추천 이유: 도심 가까이에서 색다른 분위기를 체험하고, 현지 맛집과 시장 구경까지 즐길 수 있습니다.
-

<details>
<summary>정답 보기</summary>

```python
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationSummaryMemory
from langchain_core.prompts  import ChatPromptTemplate, MessagesPlaceholder

# 모델 선언
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7)

# 메모리 생성 — 이전 대화를 요약하며 맥락 유지
memory = ConversationSummaryMemory(llm=llm, return_messages=True)

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "너는 여행 비서 트래블GPT야. "
     "사용자의 여행 루트를 기억하고, 이전 여행 내용을 바탕으로 다음 도시를 추천해줘. "
     "답변에는 반드시 '이전 여행 내용을 바탕으로 추천드리면,' 이라는 문구를 포함해야 해."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# LCEL 체인 구성
chain = prompt | llm

# 대화 시나리오
inputs = [
    "이번 주말엔 부산 갈 건데 가족 여행지 좀 추천해줘.",
    "이번엔 여수로 가볼까?",
    "그럼 마지막은 강릉이 좋을까?"
]

# 연속 대화 시뮬레이션
for user_input in inputs:
    history = memory.load_memory_variables({})["history"]
    result = chain.invoke({"history": history, "input": user_input})
    print(f"\n사용자: {user_input}\n트래블GPT: {result.content}")
    memory.save_context({"input": user_input}, {"output": result.content})

# 대화 요약 확인 (선택)
print("\n요약된 Memory Buffer:")
print(memory.buffer)
```

</details>
